In [3]:
import os
import shutil
import random
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

In [7]:
img_size = 224
batch_size = 32
num_classes = 38
epochs_cnn = 10
epochs_tl = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print (device)


cuda


In [8]:
num_classes = 38
tl_model = models.mobilenet_v2(weights="IMAGENET1K_V1")

for param in tl_model.features.parameters():
    param.requires_grad = False

tl_model.classifier[1] = nn.Linear(
    tl_model.classifier[1].in_features,
    num_classes
)

tl_model = tl_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(tl_model.classifier.parameters(), lr=1e-3)

In [12]:
# tl model Loader 
tl_model_path = "models/global_model.pth"
checkpoint = torch.load(tl_model_path, map_location=device)

model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(1280, checkpoint["num_classes"])

model = model.to(device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [14]:
import csv
import os

os.makedirs("logs", exist_ok=True)
LOG_FILE = "logs/metrics.csv"

# Write header once
with open(LOG_FILE, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["round", "global_accuracy", "num_clients"])


In [16]:
# with open(LOG_FILE, mode="a", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow([
#         round_idx,
#         global_accuracy,
#         num_selected_clients
#     ])
